In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#1. Read from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

In [0]:
display(df)

#2. Silver Transformations

###A. Trim

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

###B. Check if dates have any error or null values

In [0]:
df.select("sls_order_dt", "sls_ship_dt", "sls_due_dt").distinct().show()

In [0]:
df.filter(
    (col("sls_order_dt") == 0) |
    (length(col("sls_order_dt").cast("string")) != 8)
).show()

#### we can see irregularities in date so we need to clean that first. How?

#### we return null if date = 0 or  length of date is not 8 , otherwise we convert date into date type

In [0]:
df = df.withColumn(
        'sls_order_dt', when(
            (col('sls_order_dt') == 0) | (length(col('sls_order_dt').cast('string')) != 8), None
        ).otherwise(to_date(col('sls_order_dt').cast('string'), 'yyyyMMdd'))
    )\
    .withColumn(
        'sls_ship_dt', when(
            (col('sls_ship_dt') == 0) | (length(col('sls_ship_dt').cast('string')) != 8), None
        ).otherwise(to_date(col('sls_ship_dt').cast('string'), 'yyyyMMdd'))
    )\
    .withColumn(
        'sls_due_dt', when(
            (col('sls_due_dt') == 0) | (length(col('sls_due_dt').cast('string')) != 8), None
        ).otherwise(to_date(col('sls_due_dt').cast('string'), 'yyyyMMdd'))
    )
    

In [0]:
display(df)

###C. sales price corrections

In [0]:
df.filter((col("sls_sales") == 0) | (col('sls_quantity') == 0)).show()

In [0]:
#we dont need it yet because sales prices are never 0 but still

df = (
    df
    .withColumn(
        "sls_price",
        when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)


###D. Rename column

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(5).display()

#3. Writing Silver Table

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.crm_sales')

In [0]:
%sql
select * from workspace.silver.crm_sales limit 5